## Quality of Care Report

This report displays a compact year-level summary of quality-of-care indicators and points to generated map outputs.

In [ ]:
ROOT_PATH <- "~/workspace"
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
CODE_PATH <- file.path(ROOT_PATH, "code")
DATA_PATH <- file.path(ROOT_PATH, "data", "dhis2", "quality_of_care")
FIGURES_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care", "reporting", "outputs", "figures")

source(file.path(CODE_PATH, "snt_utils.r"))
install_and_load(c("jsonlite", "data.table", "arrow", "dplyr", "knitr", "glue", "reticulate"))

Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
openhexa <- reticulate::import("openhexa.sdk")

config_json <- jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_config.json"))
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

In [ ]:
files <- list.files(DATA_PATH, pattern = paste0("^", COUNTRY_CODE, "_quality_of_care_.*\\.parquet$"), full.names = TRUE)
if (length(files) == 0) {
  stop(glue::glue("No quality_of_care parquet found in {DATA_PATH}"))
}

latest_file <- files[which.max(file.info(files)$mtime)]
qoc <- as.data.table(arrow::read_parquet(latest_file))

summary_tbl <- qoc[, .(
  testing_rate = mean(testing_rate, na.rm = TRUE),
  treatment_rate = mean(treatment_rate, na.rm = TRUE),
  case_fatality_rate = mean(case_fatality_rate, na.rm = TRUE),
  prop_adm_malaria = mean(prop_adm_malaria, na.rm = TRUE),
  prop_malaria_deaths = mean(prop_malaria_deaths, na.rm = TRUE),
  non_malaria_all_cause_outpatients = sum(non_malaria_all_cause_outpatients, na.rm = TRUE),
  presumed_cases = sum(presumed_cases, na.rm = TRUE)
), by = .(YEAR)][order(YEAR)]

knitr::kable(summary_tbl, caption = "Quality of Care - Year-level summary")

cat(glue::glue("\nLoaded file: {latest_file}\n"))
cat(glue::glue("Map outputs folder: {FIGURES_PATH}\n"))